# Entscheidungsbaum für Klassifikation

Ein Baum zerlegt den Merkmalsraum mit Regeln wie `Merkmal <= Schwelle`. Wir untersuchen Split-Reinheit, Overfitting, Pruning und eine kleine Hyperparametersuche.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

X,y=make_moons(n_samples=700,noise=0.26,random_state=42)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
plt.scatter(X[:,0],X[:,1],c=y,cmap="RdBu_r",edgecolor="white"); plt.title("Nichtlinear trennbare Daten"); plt.show()

## Gini und ein möglicher Split

$G=1-\sum_k p_k^2$. Ein reiner Knoten hat Gini 0. Ein guter Split minimiert die nach Kindgröße gewichtete Unreinheit.

In [ ]:
def gini(labels):
    p=np.bincount(labels)/len(labels)
    return 1-np.sum(p**2)
def split_gini(X,y,feature,schwelle):
    links=X[:,feature]<=schwelle
    nach=links.mean()*gini(y[links])+(~links).mean()*gini(y[~links])
    return pd.Series({"Gini vorher":gini(y),"Gini nachher":nach,"Verbesserung":gini(y)-nach})
split_gini(X_train,y_train,1,.25).round(3)

## Hyperparameter

| Parameter | Bedeutung |
|---|---|
| `criterion` | Split-Maß: Gini, Entropie oder Log-Loss |
| `max_depth` | maximale Zahl aufeinanderfolgender Fragen |
| `min_samples_split` | Mindestgröße des Elternknotens |
| `min_samples_leaf` | Mindestgröße jedes neuen Blatts |
| `max_leaf_nodes` | maximale Zahl der Endblätter |
| `max_features` | geprüfte Merkmale je Split |
| `ccp_alpha` | Cost-Complexity-Pruning; größer ergibt kleinere Bäume |
| `class_weight` | stärkere Gewichtung seltener Klassen |

Meist stimmt man zuerst Tiefe und Blattgröße ab.

In [ ]:
def metriken(name, modell, X_test, y_test):
    pred = modell.predict(X_test)
    prob = modell.predict_proba(X_test)[:, 1]
    return {"Modell": name, "Accuracy": accuracy_score(y_test,pred),
            "Precision": precision_score(y_test,pred), "Recall": recall_score(y_test,pred),
            "F1": f1_score(y_test,pred), "ROC-AUC": roc_auc_score(y_test,prob)}

modelle={
 "Ungebremst":DecisionTreeClassifier(random_state=42),
 "Begrenzt":DecisionTreeClassifier(max_depth=4,min_samples_leaf=8,random_state=42)}
for m in modelle.values(): m.fit(X_train,y_train)
display(pd.DataFrame([metriken(n,m,X_test,y_test) for n,m in modelle.items()]).set_index("Modell").round(3))

kurve=[]
for d in range(1,16):
    m=DecisionTreeClassifier(max_depth=d,random_state=42).fit(X_train,y_train)
    kurve.append({"Tiefe":d,"F1 Training":f1_score(y_train,m.predict(X_train)),"F1 Test":f1_score(y_test,m.predict(X_test)),"F1 CV":cross_val_score(m,X_train,y_train,cv=5,scoring="f1").mean()})
pd.DataFrame(kurve).plot(x="Tiefe",marker="o",figsize=(8,4),ylim=(.75,1.01)); plt.grid(alpha=.3); plt.show()

## Kleine Grid Search

Die Suche sieht nur Trainingsdaten. Der Testdatensatz wird genau einmal zur finalen Bewertung verwendet.

In [ ]:
grid={"criterion":["gini","entropy"],"max_depth":[2,4,6,None],"min_samples_leaf":[1,5,15]}
suche=GridSearchCV(DecisionTreeClassifier(random_state=42),grid,scoring="f1",cv=5,n_jobs=-1)
suche.fit(X_train,y_train)
print("Beste Parameter:",suche.best_params_,"CV-F1:",round(suche.best_score_,3))
display(pd.DataFrame([metriken("Grid Search",suche.best_estimator_,X_test,y_test)]).set_index("Modell").round(3))
ConfusionMatrixDisplay.from_estimator(suche.best_estimator_,X_test,y_test,cmap="Blues"); plt.show()
plt.figure(figsize=(14,7)); plot_tree(suche.best_estimator_,max_depth=3,filled=True,rounded=True,feature_names=["Merkmal 1","Merkmal 2"],class_names=["0","1"]); plt.show()

**Merksatz:** Ein einzelner Baum ist gut lesbar, aber instabil. Begrenzungen und Pruning steuern seinen Bias-Varianz-Kompromiss; Skalierung ist für reine Baumsplits normalerweise nicht nötig.